In [ ]:
import os
import sys

os.chdir("..")  # run from the repo root so data/, mlflow.db and times.ttf resolve
sys.path.append(".")

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib import font_manager as fm
from omegaconf import OmegaConf
from torchvision.utils import make_grid

from data.dataset import build_datasets, make_loader
from engine.evaluate import load_checkpoint

In [ ]:
prop = fm.FontProperties(fname="times.ttf")
plt.rcParams["font.size"] = 48
os.makedirs("runs/image", exist_ok=True)

In [ ]:
CKPT = "runs:/<run_id>/checkpoints/best.pt"  # copy the run id from the MLflow UI

device = "cuda" if torch.cuda.is_available() else "cpu"
cfg = OmegaConf.create({"device": device, "tracking": {"uri": "sqlite:///mlflow.db"}})
model, ckpt, _ = load_checkpoint(cfg, CKPT)
dataset = build_datasets("data/4_infer")[0]

In [ ]:
with torch.no_grad():
    results = model(dataset[0]["low_light"].unsqueeze(0).to(device))

In [ ]:
def to_hwc_uint8(x):
    """CHW or 1CHW tensor in [0, 1] -> HWC uint8 array."""
    x = x.squeeze(0) if x.dim() == 4 else x
    return (
        (x.permute(1, 2, 0).detach().cpu().numpy() * 255).clip(0, 255).astype(np.uint8)
    )

In [ ]:
low_rgb = to_hwc_uint8(results["low_light"]["rgb"])
low_luminance = to_hwc_uint8(results["low_light"]["luminance"])
low_chroma_red = to_hwc_uint8(results["low_light"]["chroma_red"])
low_chroma_blue = to_hwc_uint8(results["low_light"]["chroma_blue"])
low_illumination = to_hwc_uint8(results["low_light"]["illumination"])
low_reflectance = to_hwc_uint8(results["low_light"]["reflectance"])
enhanced_luminance = to_hwc_uint8(results["enhanced"]["luminance"])
enhanced_chroma_red = to_hwc_uint8(results["enhanced"]["chroma_red"])
enhanced_chroma_blue = to_hwc_uint8(results["enhanced"]["chroma_blue"])
enhanced_rgb = to_hwc_uint8(results["enhanced"]["rgb"])

In [ ]:
panels = [
    (low_rgb, None, "(a) Low-light image"),
    (low_luminance, "gray", "(b) Luminance component"),
    (low_chroma_red, "gray", "(c) Cr component"),
    (low_chroma_blue, "gray", "(d) Cb component"),
    (low_illumination, "gray", "(e) Illumination component"),
    (low_reflectance, "gray", "(f) Reflectance component"),
    (enhanced_luminance, "gray", "(g) Enhanced luminance component"),
    (enhanced_chroma_red, "gray", "(h) Enhanced Cr component"),
    (enhanced_chroma_blue, "gray", "(i) Enhanced Cb component"),
    (enhanced_rgb, None, "(j) Enhanced image"),
]
plt.figure(figsize=(15, 6))
for i, (image, cmap, title) in enumerate(panels):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X=image, cmap=cmap)
    plt.title(label=title, fontproperties=prop, fontsize=14)
    plt.axis(False)
plt.tight_layout()
plt.show()

In [ ]:
def to_bgr(x):
    return cv2.cvtColor(src=x, code=cv2.COLOR_RGB2BGR)


cv2.imwrite("runs/image/00_low_x.jpg", to_bgr(low_rgb))
cv2.imwrite("runs/image/01_low_y.jpg", low_luminance)
cv2.imwrite("runs/image/02_low_cr.jpg", low_chroma_red)
cv2.imwrite("runs/image/03_low_cb.jpg", low_chroma_blue)
cv2.imwrite("runs/image/04_low_il.jpg", low_illumination)
cv2.imwrite("runs/image/05_low_re.jpg", low_reflectance)
cv2.imwrite("runs/image/06_enh_y.jpg", enhanced_luminance)
cv2.imwrite("runs/image/07_enh_cr.jpg", enhanced_chroma_red)
cv2.imwrite("runs/image/08_enh_cb.jpg", enhanced_chroma_blue)
cv2.imwrite("runs/image/09_enh_x.jpg", to_bgr(enhanced_rgb))

In [ ]:
loader = make_loader(dataset, batch_size=4, shuffle=False, num_workers=0)
batches = [batch for _, batch in zip(range(3), loader)]
with torch.no_grad():
    enhanced = [
        model(b["low_light"].to(device))["enhanced"]["rgb"].cpu() for b in batches
    ]

grids = {"Low-light images": torch.cat([b["low_light"] for b in batches])}
if dataset.paired:
    grids["Normal-light images"] = torch.cat([b["normal_light"] for b in batches])
grids["Enhanced images"] = torch.cat(enhanced)
grids = {
    k: to_hwc_uint8(make_grid(tensor=v, nrow=6, padding=2)) for k, v in grids.items()
}

for title, image in grids.items():
    plt.figure(figsize=(20, 10))
    plt.title(label=title, fontproperties=prop, fontsize=14)
    plt.imshow(X=image)
    plt.axis(False)
plt.tight_layout()
plt.show()

In [ ]:
cv2.imwrite("runs/image/10_grid_low_light.jpg", to_bgr(grids["Low-light images"]))
cv2.imwrite("runs/image/11_grid_enhanced.jpg", to_bgr(grids["Enhanced images"]))